In [81]:
# 1) Setup and Data Loading

from pathlib import Path
import json
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)

ROOT = Path("../problem_statement")
DATA_DIR = ROOT / "data"
ANNOT_DIR = DATA_DIR / "annotations"


def read_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

logs = read_jsonl(DATA_DIR / "production_logs.jsonl")
outcomes = read_jsonl(DATA_DIR / "outcomes.jsonl")
a1 = read_jsonl(ANNOT_DIR / "annotator_1.jsonl")
a2 = read_jsonl(ANNOT_DIR / "annotator_2.jsonl")
a3 = read_jsonl(ANNOT_DIR / "annotator_3.jsonl")

print(f"logs: {len(logs)}")
print(f"outcomes: {len(outcomes)}")
print(f"annotator_1: {len(a1)}")
print(f"annotator_2: {len(a2)}")
print(f"annotator_3: {len(a3)}")

logs: 700
outcomes: 700
annotator_1: 200
annotator_2: 200
annotator_3: 200


In [82]:
logs[0]

# for k, v in logs[0].items():
#     if isinstance(v, list):
#         print(k, len(v))
#     else:
#         print(k, type(v), v)


{'conversation_id': '192f029c-2626-7e25-7fee-3fff275530b7',
 'messages': [{'role': 'bot',
   'text': "Hi Arjun! This is Priya calling from Riverline Financial Services. Hope you're doing well today 😊",
   'timestamp': '2026-01-21T09:01:00',
   'turn': 0},
  {'role': 'borrower',
   'text': 'Hello Priya, yes I was actually expecting someone to reach out. I want to get this sorted out.',
   'timestamp': '2026-01-21T09:02:06',
   'turn': 1},
  {'role': 'bot',
   'text': "That's great to hear! Just need to quickly verify your details before we proceed. Could you confirm the last 4 digits of your registered phone number?",
   'timestamp': '2026-01-21T09:02:10',
   'turn': 2},
  {'role': 'borrower',
   'text': 'Sure, and my registered email is rajesh@email.com if that helps too.',
   'timestamp': '2026-01-21T09:03:39',
   'turn': 3},
  {'role': 'bot',
   'text': "Thanks Arjun! So I'm reaching out about your outstanding balance of ₹1,65,000 on your account. How were you thinking of resolving t

In [ ]:
outcomes[0]

In [ ]:
a1[0]

In [ ]:
# 2) Quick Schema Peek (one sample row from each file)

print("logs keys:")
for k in logs[0].keys():
    print("  ", k)
print("outcomes keys:")
for k in outcomes[0].keys():
    print("  ", k)
print("annotation keys:")
for k in a1[0].keys():
    print("  ", k)

print("\nSample log conversation_id:", logs[0]["conversation_id"])
print("Messages in sample:", len(logs[0]["messages"]))
print("Transitions in sample:", len(logs[0]["state_transitions"]))
print("Function calls in sample:", len(logs[0]["function_calls"]))

In [ ]:
# 3) Build Flat DataFrames (conversation-level)

logs_df = pd.DataFrame([
    {
        "conversation_id": c["conversation_id"],
        "n_messages": len(c.get("messages", [])),
        "n_borrower_msgs": sum(m.get("role") == "borrower" for m in c.get("messages", [])),
        "n_bot_msgs": sum(m.get("role") == "bot" for m in c.get("messages", [])),
        "n_classifications": len(c.get("bot_classifications", [])),
        "n_transitions": len(c.get("state_transitions", [])),
        "n_function_calls": len(c.get("function_calls", [])),
        "language": c.get("metadata", {}).get("language"),
        "zone": c.get("metadata", {}).get("zone"),
        "dpd": c.get("metadata", {}).get("dpd"),
        "pos": c.get("metadata", {}).get("pos"),
        "tos": c.get("metadata", {}).get("tos"),
        "total_turns_meta": c.get("metadata", {}).get("total_turns"),
    }
    for c in logs
])

outcomes_df = pd.DataFrame(outcomes)

a1_df = pd.DataFrame(a1).assign(annotator="annotator_1")
a2_df = pd.DataFrame(a2).assign(annotator="annotator_2")
a3_df = pd.DataFrame(a3).assign(annotator="annotator_3")
ann_df = pd.concat([a1_df, a2_df, a3_df], ignore_index=True)

print("logs_df:", logs_df.shape)
print("outcomes_df:", outcomes_df.shape)
print("ann_df:", ann_df.shape)
logs_df.head(3)

In [ ]:
# 4) Basic Missingness and Data Health Checks

print("=== logs_df missingness ===")
display(logs_df.isna().mean().sort_values(ascending=False).to_frame("missing_rate").head(15))

print("=== outcomes_df missingness ===")
display(outcomes_df.isna().mean().sort_values(ascending=False).to_frame("missing_rate").head(15))

print("=== quick sanity checks ===")
print("Rows in logs:", len(logs_df), "| unique conversation_id:", logs_df["conversation_id"].nunique())
print("Rows in outcomes:", len(outcomes_df), "| unique conversation_id:", outcomes_df["conversation_id"].nunique())
print("Rows in ann_df:", len(ann_df), "| unique conversation_id:", ann_df["conversation_id"].nunique())

In [ ]:
# 5) Conversation-Level Distributions (Language, Zone, DPD, Turn Counts)

print("=== language distribution ===")
display(logs_df["language"].value_counts(dropna=False).to_frame("count"))

print("=== zone distribution ===")
display(logs_df["zone"].value_counts(dropna=False).to_frame("count"))

print("=== DPD summary ===")
display(logs_df["dpd"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

print("=== message count summary ===")
display(logs_df[["n_messages", "n_borrower_msgs", "n_bot_msgs", "n_transitions", "n_function_calls"]].describe())

In [ ]:
# 6) Classification Distribution and Confidence

classification_counter = Counter()
confidence_counter = Counter()

for c in logs:
    for bc in c.get("bot_classifications", []):
        classification_counter[bc.get("classification")] += 1
        confidence_counter[bc.get("confidence")] += 1

cls_df = pd.DataFrame(classification_counter.items(), columns=["classification", "count"]).sort_values("count", ascending=False)
conf_df = pd.DataFrame(confidence_counter.items(), columns=["confidence", "count"]).sort_values("count", ascending=False)

print("=== classification counts ===")
display(cls_df)

print("=== confidence counts ===")
display(conf_df)

In [ ]:
# 7) State Transition Exploration (most common transitions)

transition_counter = Counter()
state_counter = Counter()

for c in logs:
    for t in c.get("state_transitions", []):
        frm = t.get("from_state")
        to = t.get("to_state")
        transition_counter[(frm, to)] += 1
        state_counter[frm] += 1
        state_counter[to] += 1

transition_df = pd.DataFrame(
    [{"from_state": k[0], "to_state": k[1], "count": v} for k, v in transition_counter.items()]
).sort_values("count", ascending=False)

print("=== top transitions ===")
display(transition_df.head(25))

print("=== state frequency (from+to appearances) ===")
display(pd.DataFrame(state_counter.items(), columns=["state", "count"]).sort_values("count", ascending=False))

In [ ]:
# 8) Function Call Exploration

fn_counter = Counter()
fn_by_conv = []

for c in logs:
    fn_list = [f.get("function") for f in c.get("function_calls", [])]
    for fn in fn_list:
        fn_counter[fn] += 1
    fn_by_conv.append({
        "conversation_id": c["conversation_id"],
        "n_functions": len(fn_list),
        "functions": fn_list,
    })

fn_df = pd.DataFrame(fn_by_conv)
print("=== function frequency ===")
display(pd.DataFrame(fn_counter.items(), columns=["function", "count"]).sort_values("count", ascending=False))

print("=== conversations with no function calls ===")
print((fn_df["n_functions"] == 0).sum())

In [ ]:
# 9) Join Logs + Outcomes and Inspect Outcome Rates

analysis_df = logs_df.merge(outcomes_df, on="conversation_id", how="left", validate="one_to_one")

print("Joined shape:", analysis_df.shape)

for col in ["payment_received", "borrower_complained", "regulatory_flag", "required_human_intervention"]:
    if col in analysis_df.columns:
        print(f"\n{col} value counts:")
        display(analysis_df[col].value_counts(dropna=False).to_frame("count"))

if "channel_attribution" in analysis_df.columns:
    print("\nchannel_attribution:")
    display(analysis_df["channel_attribution"].value_counts(dropna=False).to_frame("count"))

In [ ]:
# 10) Outcome Slices by Segment (language, zone, dpd bucket)

seg_df = analysis_df.copy()
seg_df["dpd_bucket"] = pd.cut(seg_df["dpd"], bins=[-1, 30, 60, 90, 9999], labels=["0-30", "31-60", "61-90", "90+"])

for seg_col in ["language", "zone", "dpd_bucket"]:
    if seg_col in seg_df.columns:
        grp = seg_df.groupby(seg_col, dropna=False).agg(
            n=("conversation_id", "count"),
            complaint_rate=("borrower_complained", "mean"),
            regulatory_rate=("regulatory_flag", "mean"),
            payment_rate=("payment_received", "mean"),
        ).sort_values("n", ascending=False)
        print(f"\n=== Segment: {seg_col} ===")
        display(grp)

In [ ]:
# 11) Annotation Coverage and Overlap

ann_cov = ann_df.groupby("annotator")["conversation_id"].nunique().to_frame("unique_conversations")
print("=== coverage per annotator ===")
display(ann_cov)

ids1 = set(a1_df["conversation_id"])
ids2 = set(a2_df["conversation_id"])
ids3 = set(a3_df["conversation_id"])

print("Overlap a1 & a2:", len(ids1 & ids2))
print("Overlap a1 & a3:", len(ids1 & ids3))
print("Overlap a2 & a3:", len(ids2 & ids3))
print("Triple overlap:", len(ids1 & ids2 & ids3))

In [ ]:
# 12) Annotator Disagreement on Quality Score (pairwise)

pair12 = a1_df[["conversation_id", "quality_score"]].merge(
    a2_df[["conversation_id", "quality_score"]], on="conversation_id", suffixes=("_a1", "_a2")
)
pair13 = a1_df[["conversation_id", "quality_score"]].merge(
    a3_df[["conversation_id", "quality_score"]], on="conversation_id", suffixes=("_a1", "_a3")
)
pair23 = a2_df[["conversation_id", "quality_score"]].merge(
    a3_df[["conversation_id", "quality_score"]], on="conversation_id", suffixes=("_a2", "_a3")
)

def pair_stats(df, c1, c2):
    return {
        "n_overlap": len(df),
        "mae": (df[c1] - df[c2]).abs().mean(),
        "corr": df[c1].corr(df[c2]),
    }

stats = pd.DataFrame([
    {"pair": "a1_vs_a2", **pair_stats(pair12, "quality_score_a1", "quality_score_a2")},
    {"pair": "a1_vs_a3", **pair_stats(pair13, "quality_score_a1", "quality_score_a3")},
    {"pair": "a2_vs_a3", **pair_stats(pair23, "quality_score_a2", "quality_score_a3")},
])

display(stats)

In [ ]:
# 13) Failure Category Frequency from Annotations

failure_rows = []
for _, row in ann_df.iterrows():
    fps = row.get("failure_points", []) or []
    for fp in fps:
        failure_rows.append({
            "conversation_id": row["conversation_id"],
            "annotator": row["annotator"],
            "category": fp.get("category"),
            "severity": fp.get("severity"),
            "turn": fp.get("turn"),
        })

failure_df = pd.DataFrame(failure_rows)
print("failure rows:", failure_df.shape)

display(
    failure_df.groupby("category", dropna=False)
    .agg(n=("category", "count"), mean_severity=("severity", "mean"))
    .sort_values("n", ascending=False)
)

In [ ]:
# 14) Quick Rule-Mining Helpers (DNC / Hardship keyword spotting)

DNC_PATTERNS = [
    r"\bstop\b",
    r"do not contact",
    r"don't contact",
    r"leave me alone",
    r"block me",
]

HARDSHIP_PATTERNS = [
    r"lost my job",
    r"medical",
    r"hospital",
    r"family emergency",
    r"can't pay",
    r"financial problem",
]


def text_has_any_pattern(text, patterns):
    txt = (text or "").lower()
    return any(re.search(p, txt) for p in patterns)

sample_hits = []
for c in logs[:80]:  # small sample first
    for m in c.get("messages", []):
        if m.get("role") != "borrower":
            continue
        txt = m.get("text", "")
        if text_has_any_pattern(txt, DNC_PATTERNS):
            sample_hits.append((c["conversation_id"], m.get("turn"), "DNC", txt))
        if text_has_any_pattern(txt, HARDSHIP_PATTERNS):
            sample_hits.append((c["conversation_id"], m.get("turn"), "HARDSHIP", txt))

sample_hits[:20]

In [ ]:
# 15) Pull 3 High-Signal Conversations for Manual Review

# Heuristic: complaint/regulatory OR very long conversation OR many function calls
candidate_df = analysis_df.copy()

candidate_df["flag_signal"] = (
    candidate_df.get("borrower_complained", False).fillna(False).astype(int)
    + candidate_df.get("regulatory_flag", False).fillna(False).astype(int)
)

candidate_df = candidate_df.sort_values(
    ["flag_signal", "n_messages", "n_function_calls"], ascending=[False, False, False]
)

top_ids = candidate_df["conversation_id"].head(3).tolist()
print("Top conversation_ids to inspect manually:", top_ids)

for cid in top_ids:
    conv = next(c for c in logs if c["conversation_id"] == cid)
    print("\n", "=" * 80)
    print("conversation_id:", cid)
    print("metadata:", conv.get("metadata", {}))
    print("-- first 8 messages --")
    for m in conv.get("messages", [])[:8]:
        print(f"turn={m.get('turn')} | {m.get('role')}: {m.get('text')}")

In [ ]:
logs[0]

In [ ]:
# 16) Unified Conversation Bundle by conversation_id

# Build fast lookup maps
log_map = {x["conversation_id"]: x for x in logs}
outcome_map = {x["conversation_id"]: x for x in outcomes}
a1_map = {x["conversation_id"]: x for x in a1}
a2_map = {x["conversation_id"]: x for x in a2}
a3_map = {x["conversation_id"]: x for x in a3}

all_ids = sorted(set(log_map) | set(outcome_map) | set(a1_map) | set(a2_map) | set(a3_map))

conv_bundle = {}
for cid in all_ids:
    rec = {
        "conversation_id": cid,
        "metadata": log_map.get(cid, {}).get("metadata", {}),
        "prod_log": log_map.get(cid),
        "outcome": outcome_map.get(cid),
        "an1": a1_map.get(cid),
        "an2": a2_map.get(cid),
        "an3": a3_map.get(cid),
    }
    conv_bundle[cid] = rec

print("Total unified conversations:", len(conv_bundle))
import pandas as pd

# Only flatten and export these 7 columns:
# conversation_id, prod_log, outcome, an1, an2, an3, metadata

conv_bundle_flat = []
for cid, v in conv_bundle.items():
    # Compute "no of annotations" for each conversation:
    n_anns = 0
    for field in ("an1", "an2", "an3"):
        if v.get(field) is not None:
            n_anns += 1
    v["n_annotations"] = n_anns
    flat = {
        "conversation_id": cid,
        "n_annotations": n_anns,
        "metadata": v.get("metadata", {}),
        "prod_log": v.get("prod_log"),
        "outcome": v.get("outcome"),
        "an1": v.get("an1"),
        "an2": v.get("an2"),
        "an3": v.get("an3"),
    }
    conv_bundle_flat.append(flat)

df_conv_bundle = pd.DataFrame(conv_bundle_flat)
df_conv_bundle.to_csv("conversation_bundle_flat.csv", index=False)
print("Saved conv_bundle to conversation_bundle_flat.csv")


# # Flat summary frame for quick analysis
# bundle_summary_df = pd.DataFrame([
#     {
#         "conversation_id": cid,
#         "has_prod_log": v["has_prod_log"],
#         "has_outcome": v["has_outcome"],
#         "has_an1": v["an1"] is not None,
#         "has_an2": v["an2"] is not None,
#         "has_an3": v["an3"] is not None,
#         "n_annotations": v["n_annotations"],
#         "is_triple_labeled": v["is_triple_labeled"],
#         "language": (v["prod_log"] or {}).get("metadata", {}).get("language"),
#         "zone": (v["prod_log"] or {}).get("metadata", {}).get("zone"),
#         "dpd": (v["prod_log"] or {}).get("metadata", {}).get("dpd"),
#         "borrower_complained": (v["outcome"] or {}).get("borrower_complained"),
#         "regulatory_flag": (v["outcome"] or {}).get("regulatory_flag"),
#         "payment_received": (v["outcome"] or {}).get("payment_received"),
#     }
#     for cid, v in conv_bundle.items()
# ])

# print("\nCoverage view:")
# display(bundle_summary_df[["has_prod_log", "has_outcome", "n_annotations", "is_triple_labeled"]].describe(include="all"))

# print("\nAnnotation count distribution:")
# display(bundle_summary_df["n_annotations"].value_counts().sort_index().to_frame("count"))

# bundle_summary_df.head(10)

In [ ]:
annotation_counts = df_conv_bundle["n_annotations"].value_counts().sort_index()
print("Number of IDs for each value of n_annotations:")
print(annotation_counts)

In [ ]:
flat

In [ ]:
df_conv_bundle = pd.DataFrame(conv_bundle_flat)
df_conv_bundle.to_csv("conversation_bundle_flat.csv", index=False)
print("Saved conv_bundle to conversation_bundle_flat.csv")

In [ ]:
# 17) One-View Conversation Visualizer (by conversation_id)

import html
from IPython.display import display, HTML


def _safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def _ensure_conv_bundle():
    """Ensure conv_bundle exists in notebook scope."""
    global conv_bundle
    if "conv_bundle" in globals() and isinstance(conv_bundle, dict) and len(conv_bundle) > 0:
        return conv_bundle

    log_map = {x["conversation_id"]: x for x in logs}
    outcome_map = {x["conversation_id"]: x for x in outcomes}
    a1_map = {x["conversation_id"]: x for x in a1}
    a2_map = {x["conversation_id"]: x for x in a2}
    a3_map = {x["conversation_id"]: x for x in a3}

    all_ids = sorted(set(log_map) | set(outcome_map) | set(a1_map) | set(a2_map) | set(a3_map))

    conv_bundle = {}
    for cid in all_ids:
        conv_bundle[cid] = {
            "conversation_id": cid,
            "prod_log": log_map.get(cid),
            "outcome": outcome_map.get(cid),
            "an1": a1_map.get(cid),
            "an2": a2_map.get(cid),
            "an3": a3_map.get(cid),
        }
    return conv_bundle


def _turn_index(prod_log: dict) -> pd.DataFrame:
    messages = prod_log.get("messages", []) or []
    classifications = prod_log.get("bot_classifications", []) or []
    transitions = prod_log.get("state_transitions", []) or []
    function_calls = prod_log.get("function_calls", []) or []

    # Index helper maps by turn (allow multiple transitions/function calls per turn)
    class_by_turn = {}
    for c in classifications:
        class_by_turn[c.get("turn")] = {
            "classification": c.get("classification"),
            "confidence": c.get("confidence"),
            "input_text": c.get("input_text"),
        }

    trans_by_turn = {}
    for t in transitions:
        turn = t.get("turn")
        trans_by_turn.setdefault(turn, []).append(f"{t.get('from_state')} → {t.get('to_state')}")

    fn_by_turn = {}
    for f in function_calls:
        turn = f.get("turn")
        params = f.get("params", {})
        fn_by_turn.setdefault(turn, []).append(f"{f.get('function')}({params})")

    rows = []
    for m in messages:
        turn = m.get("turn")
        cls = class_by_turn.get(turn, {}) if m.get("role") == "borrower" else {}
        rows.append(
            {
                "turn": turn,
                "timestamp": m.get("timestamp"),
                "role": m.get("role"),
                "text": m.get("text", ""),
                "classification": cls.get("classification"),
                "confidence": cls.get("confidence"),
                "transitions": " | ".join(trans_by_turn.get(turn, [])),
                "function_calls": " | ".join(fn_by_turn.get(turn, [])),
            }
        )

    df = pd.DataFrame(rows).sort_values(["turn", "timestamp"], kind="stable").reset_index(drop=True)
    return df


def render_conversation(conversation_id: str, max_text_chars: int = 170):
    """
    Render one compact, all-in-one conversation view.

    Shows:
    - Header with key metadata + outcome snapshot
    - Turn-level timeline (role, text, classification, transitions, function calls)
    - Footer with full metadata/outcome/annotation details
    """
    # Static mapping for state -> id, from image
    STATE_ID_MAP = {
        "new": 0,
        "message_received": 1,
        "verification": 2,
        "intent_asked": 3,
        "settlement_explained": 4,
        "amount_pending": 5,
        "amount_sent": 6,
        "date_amount_asked": 7,
        "payment_confirmed": 8,
    }
    # Helper to insert id into any state name occurrences
    def _annotate_state(state_name):
        if not isinstance(state_name, str):
            return state_name
        clean = state_name.strip()
        sid = STATE_ID_MAP.get(clean)
        if sid is not None:
            return f"{clean}[{sid}]"
        return clean

    def _annotate_transitions(transitions_str):
        """Expands 'stateA → stateB | stateC → stateD' to new[0]\n→\nmessage_received[1] | ... (one per line)."""
        if not transitions_str:
            return ""
        # Handle multiple transitions separated by '|'
        parts = [t.strip() for t in transitions_str.split('|')]
        expanded = []
        for part in parts:
            if '→' in part:
                src, tgt = [s.strip() for s in part.split('→', 1)]
                src_anno = _annotate_state(src)
                tgt_anno = _annotate_state(tgt)
                expanded.append(f"{src_anno}\n→\n{tgt_anno}")
            else:
                expanded.append(_annotate_state(part))
        return ' | '.join(expanded)
    
    bundle = _ensure_conv_bundle()
    row = bundle.get(conversation_id)

    if row is None:
        print(f"conversation_id not found: {conversation_id}")
        return

    prod_log = row.get("prod_log")
    outcome = row.get("outcome")
    an1_row, an2_row, an3_row = row.get("an1"), row.get("an2"), row.get("an3")

    if prod_log is None:
        print(f"No prod_log for conversation_id: {conversation_id}")
        return

    meta = prod_log.get("metadata", {}) or {}
    turn_df = _turn_index(prod_log)

    # Compact text for table view
    def compact_text(t):

        t = "" if t is None else str(t)
        # return t if len(t) <= max_text_chars else t[: max_text_chars - 3] + "..."
        return t

    view_df = turn_df.copy()
    view_df["text"] = view_df["text"].map(compact_text)
    # Add state id annotations + vertical layout to transitions col
    view_df["transitions"] = view_df["transitions"].map(_annotate_transitions)

    # Pretty chips in HTML
    def chip(text, bg="#eef2ff", fg="#1e3a8a"):
        if text is None or text == "":
            return ""
        return (
            f"<span style='display:inline-block;padding:2px 8px;border-radius:999px;"
            f"background:{bg};color:{fg};font-size:11px;margin-right:4px'>{html.escape(str(text))}</span>"
        )

    header_html = f"""
    <div style='border:1px solid #d1d5db;border-radius:10px;padding:12px 14px;margin:8px 0 12px 0;'>
      <div style='font-size:18px;font-weight:700;margin-bottom:6px;'>Conversation: {html.escape(conversation_id)}</div>
      <div style='font-size:13px;line-height:1.6;'>
        <b>Language:</b> {html.escape(str(meta.get('language')))} &nbsp;|&nbsp;
        <b>Zone:</b> {html.escape(str(meta.get('zone')))} &nbsp;|&nbsp;
        <b>DPD:</b> {html.escape(str(meta.get('dpd')))} &nbsp;|&nbsp;
        <b>POS/TOS:</b> {html.escape(str(meta.get('pos')))} / {html.escape(str(meta.get('tos')))} &nbsp;|&nbsp;
        <b>Total Turns(meta):</b> {html.escape(str(meta.get('total_turns')))}
      </div>
      <div style='font-size:13px;line-height:1.6;margin-top:6px;'>
        <b>Outcome:</b>
        {chip(f"payment_received={_safe_get(outcome or {}, 'payment_received')}", '#dcfce7', '#166534')}
        {chip(f"complained={_safe_get(outcome or {}, 'borrower_complained')}", '#fee2e2', '#991b1b')}
        {chip(f"regulatory_flag={_safe_get(outcome or {}, 'regulatory_flag')}", '#fde68a', '#92400e')}
        {chip(f"channel_attribution={_safe_get(outcome or {}, 'channel_attribution')}", '#e0f2fe', '#075985')}
      </div>
    </div>
    """
    display(HTML(header_html))

    def role_style(role):
        if role == "bot":
            return "background:#eef6ff;color:#1d4ed8;font-weight:600;"
        if role == "borrower":
            return "background:#f0fdf4;color:#166534;font-weight:600;"
        return ""

    # Table column widths: text col = 50%, others shrink (allow text to go multi-line, reduce scroll)
    # We'll also display transitions chips with white-space:pre-line, so line breaks work for state->id
    rows_html = []
    for _, r in view_df.iterrows():
        cls_chip = chip(
            f"{r['classification']} ({r['confidence']})" if pd.notna(r["classification"]) else "",
            "#ede9fe",
            "#5b21b6",
        )
        fn_chip = chip(r["function_calls"], "#ffe4e6", "#9f1239") if r["function_calls"] else ""
        tr_chip = chip(r["transitions"], "#ecfccb", "#3f6212") if r["transitions"] else ""

        rows_html.append(
            "<tr>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;text-align:center;width:4%;min-width:38px;'>{r['turn']}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;{role_style(r['role'])}text-align:center;border-radius:6px;width:5%;min-width:60px;'>{html.escape(str(r['role']))}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;font-size:12px;color:#6b7280;width:10%;min-width:100px;'>{html.escape(str(r['timestamp']))}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;width:50%;max-width:0;word-break:break-word;font-size:13px;'>{html.escape(str(r['text']))}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;min-width:100px;width:10%'>{cls_chip}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;min-width:100px;width:11%;white-space:pre-line;vertical-align:top'>{tr_chip}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;min-width:120px;width:10%'>{fn_chip}</td>"
            "</tr>"
        )

    table_html = f"""
    <div style='border:1px solid #d1d5db;border-radius:10px;overflow:auto;margin-top:8px;'>
      <table style='border-collapse:collapse;width:100%;table-layout:fixed;font-size:13px;'>
        <thead style='background:#f9fafb;'>
          <tr>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:4%;min-width:38px;'>Turn</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:5%;min-width:60px;'>Role</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:10%;min-width:100px;'>Timestamp</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:50%;max-width:0;'>Text</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:10%;min-width:100px;'>Classification</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:11%;min-width:100px;'>State<br/>Transition(s)<br/><span style="font-weight:400;font-size:11px;">(with state id)</span></th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:10%;min-width:120px;'>Function<br/>Call(s)</th>
          </tr>
        </thead>
        <tbody>
          {''.join(rows_html)}
        </tbody>
      </table>
    </div>
    """
    display(HTML(table_html))

    # Footer JSON blocks (metadata/outcome/annotations)
    def short_anno(a):
        if a is None:
            return None
        return {
            "quality_score": a.get("quality_score"),
            "risk_flags": a.get("risk_flags"),
            "overall_assessment": a.get("overall_assessment"),
            "failure_points_count": len(a.get("failure_points", []) or []),
        }

    footer = {
        "metadata": meta,
        "outcome": outcome,
        "annotations": {
            "an1": short_anno(an1_row),
            "an2": short_anno(an2_row),
            "an3": short_anno(an3_row),
        },
    }

    display(HTML("<div style='margin-top:10px;font-weight:700'>Metadata + Outcome + Annotations</div>"))
    display(footer)


def render_conversation_from_df(df: pd.DataFrame, conversation_id: str):
    """
    Convenience helper: fetch from a flat df by conversation_id, then render.
    Useful when browsing IDs from bundle_summary_df / analysis_df.
    """
    if "conversation_id" not in df.columns:
        raise ValueError("DataFrame must contain a 'conversation_id' column")
    if conversation_id not in set(df["conversation_id"].astype(str)):
        raise ValueError(f"conversation_id {conversation_id} not found in given DataFrame")
    render_conversation(conversation_id)

print("Visualizer ready: use render_conversation('<conversation_id>')")

In [ ]:
# 18) Example usage

# Replace with any conversation_id you want to inspect
example_cid = "ffc79a4b-b8f6-eb22-2a52-ba3352e6e16c"
example_cid = "051670b7-b772-dc86-433a-099a40b8ab18"
render_conversation(example_cid)

In [ ]:
# 19) Clean Wide-Table Visualizer (With Improved Wrapping and Responsive Table/Fix Overflow)

import html
from IPython.display import display, HTML


STATE_ID_MAP = {
    "new": 0,
    "message_received": 1,
    "verification": 2,
    "intent_asked": 3,
    "settlement_explained": 4,
    "amount_pending": 5,
    "amount_sent": 6,
    "date_amount_asked": 7,
    "payment_confirmed": 8,
    "escalated": 9,
    "dormant": 10,
}


def _safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def _ensure_conv_bundle_v2():
    global conv_bundle
    if "conv_bundle" in globals() and isinstance(conv_bundle, dict) and len(conv_bundle) > 0:
        return conv_bundle

    log_map = {x["conversation_id"]: x for x in logs}
    outcome_map = {x["conversation_id"]: x for x in outcomes}
    a1_map = {x["conversation_id"]: x for x in a1}
    a2_map = {x["conversation_id"]: x for x in a2}
    a3_map = {x["conversation_id"]: x for x in a3}

    all_ids = sorted(set(log_map) | set(outcome_map) | set(a1_map) | set(a2_map) | set(a3_map))
    conv_bundle = {}
    for cid in all_ids:
        conv_bundle[cid] = {
            "conversation_id": cid,
            "prod_log": log_map.get(cid),
            "outcome": outcome_map.get(cid),
            "an1": a1_map.get(cid),
            "an2": a2_map.get(cid),
            "an3": a3_map.get(cid),
        }
    return conv_bundle


def _annotate_state(state_name):
    if not isinstance(state_name, str):
        return str(state_name)
    s = state_name.strip()
    sid = STATE_ID_MAP.get(s)
    return f"{s}[{sid}]" if sid is not None else s


def _format_transition_list(transitions):
    if not transitions:
        return ""
    out = []
    for t in transitions:
        src = _annotate_state(t.get("from_state"))
        tgt = _annotate_state(t.get("to_state"))
        out.append(f"{src} -> {tgt}")
    return "<br>".join(html.escape(x) for x in out)


def _format_function_list(functions):
    if not functions:
        return ""
    lines = []
    for f in functions:
        fn = f.get("function")
        params = f.get("params", {})
        lines.append(f"{fn}({params})")
    return "<br>".join(html.escape(x) for x in lines)


def _build_turn_rows(prod_log):
    messages = prod_log.get("messages", []) or []
    classifications = prod_log.get("bot_classifications", []) or []
    transitions = prod_log.get("state_transitions", []) or []
    function_calls = prod_log.get("function_calls", []) or []

    cls_by_turn = {}
    for c in classifications:
        cls_by_turn[c.get("turn")] = c

    tr_by_turn = {}
    for t in transitions:
        tr_by_turn.setdefault(t.get("turn"), []).append(t)

    fn_by_turn = {}
    for f in function_calls:
        fn_by_turn.setdefault(f.get("turn"), []).append(f)

    rows = []
    for m in sorted(messages, key=lambda x: (x.get("turn", -1), x.get("timestamp", ""))):
        turn = m.get("turn")
        role = m.get("role")
        c = cls_by_turn.get(turn, {}) if role == "borrower" else {}

        rows.append(
            {
                "turn": turn,
                "role": role,
                "timestamp": m.get("timestamp"),
                "text": m.get("text", ""),
                "classification": c.get("classification"),
                "confidence": c.get("confidence"),
                "transitions_html": _format_transition_list(tr_by_turn.get(turn, [])),
                "functions_html": _format_function_list(fn_by_turn.get(turn, [])),
            }
        )
    return rows


def _chip(text, bg="#eef2ff", fg="#1e3a8a"):
    if text is None or text == "":
        return ""
    return (
        f"<span style='display:inline-block;padding:2px 8px;border-radius:999px;"
        f"background:{bg};color:{fg};font-size:11px;margin-right:6px'>{html.escape(str(text))}</span>"
    )


def _render_kv_table(title, data_dict):
    if data_dict is None:
        display(HTML(f"<div style='margin-top:8px'><b>{html.escape(title)}:</b> None</div>"))
        return

    rows = []
    for k, v in data_dict.items():
        rows.append(
            "<tr>"
            f"<td style='padding:6px 8px;border-bottom:1px solid #e5e7eb;font-weight:600;width:220px'>{html.escape(str(k))}</td>"
            f"<td style='padding:6px 8px;border-bottom:1px solid #e5e7eb'>{html.escape(str(v))}</td>"
            "</tr>"
        )
    table = (
        f"<div style='margin-top:10px'><b>{html.escape(title)}</b></div>"
        "<div style='border:1px solid #d1d5db;border-radius:8px;overflow:auto;'>"
        "<table style='width:100%;border-collapse:collapse;font-size:13px;'>"
        f"{''.join(rows)}"
        "</table></div>"
    )
    display(HTML(table))


def render_conversation_clean(conversation_id: str):
    """Simple clean one-view renderer by conversation_id."""
    bundle = _ensure_conv_bundle_v2()
    row = bundle.get(conversation_id)

    if row is None:
        print(f"conversation_id not found: {conversation_id}")
        return

    prod_log = row.get("prod_log")
    outcome = row.get("outcome")
    an1_row, an2_row, an3_row = row.get("an1"), row.get("an2"), row.get("an3")

    if prod_log is None:
        print(f"No prod_log for conversation_id: {conversation_id}")
        return

    meta = prod_log.get("metadata", {}) or {}
    rows = _build_turn_rows(prod_log)

    # Header
    header = f"""
    <div style='border:1px solid #d1d5db;border-radius:10px;padding:12px 14px;margin:8px 0 12px 0;'>
      <div style='font-size:18px;font-weight:700;margin-bottom:6px;'>Conversation: {html.escape(conversation_id)}</div>
      <div style='font-size:13px;line-height:1.7;'>
        <b>Language:</b> {html.escape(str(meta.get('language')))} &nbsp;|&nbsp;
        <b>Zone:</b> {html.escape(str(meta.get('zone')))} &nbsp;|&nbsp;
        <b>DPD:</b> {html.escape(str(meta.get('dpd')))} &nbsp;|&nbsp;
        <b>POS/TOS:</b> {html.escape(str(meta.get('pos')))} / {html.escape(str(meta.get('tos')))} &nbsp;|&nbsp;
        <b>Total Turns(meta):</b> {html.escape(str(meta.get('total_turns')))}
      </div>
      <div style='font-size:13px;line-height:1.7;margin-top:6px;'>
        <b>Outcome:</b>
        {_chip(f"payment_received={_safe_get(outcome or {}, 'payment_received')}", '#dcfce7', '#166534')}
        {_chip(f"complained={_safe_get(outcome or {}, 'borrower_complained')}", '#fee2e2', '#991b1b')}
        {_chip(f"regulatory_flag={_safe_get(outcome or {}, 'regulatory_flag')}", '#fde68a', '#92400e')}
        {_chip(f"channel_attribution={_safe_get(outcome or {}, 'channel_attribution')}", '#e0f2fe', '#075985')}
      </div>
    </div>
    """
    display(HTML(header))

    # State legend
    legend_text = ", ".join([f"{k}[{v}]" for k, v in STATE_ID_MAP.items()])
    display(HTML(f"<div style='font-size:12px;margin:4px 0 10px 0;'><b>State ID Legend:</b> {html.escape(legend_text)}</div>"))

    def role_badge(role):
        if role == "bot":
            return "<span style='padding:2px 8px;border-radius:999px;background:#e0ecff;color:#1d4ed8;font-weight:600'>bot</span>"
        if role == "borrower":
            return "<span style='padding:2px 8px;border-radius:999px;background:#e8f8ec;color:#166534;font-weight:600'>borrower</span>"
        return html.escape(str(role))

    # Main table (text-first)
    row_html = []
    for r in rows:
        cls_text = ""
        if r.get("classification"):
            cls_text = f"{r['classification']} ({r.get('confidence')})"

        row_html.append(
            "<tr>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;width:9%;white-space:nowrap'>{html.escape(str(r['turn']))}<br>{role_badge(r['role'])}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;width:11%;font-size:12px;color:#6b7280'>{html.escape(str(r['timestamp']))}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;width:50%;word-break:break-word;line-height:1.4'>{html.escape(str(r['text']))}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;width:8%;font-size:12px'>{html.escape(cls_text)}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;width:10%;font-size:12px;line-height:1.4'>{r['transitions_html']}</td>"
            f"<td style='padding:8px;border-bottom:1px solid #e5e7eb;width:22%;font-size:12px;line-height:1.4'>{r['functions_html']}</td>"
            "</tr>"
        )

    table = f"""
    <div style='border:1px solid #d1d5db;border-radius:10px;overflow:auto;'>
      <table style='border-collapse:collapse;width:100%;table-layout:fixed;font-size:13px;'>
        <thead style='background:#f9fafb;'>
          <tr>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:9%'>Turn/Role</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:11%'>Timestamp</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:62%'>Text</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:8%'>Classification</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:10%'>Transition(s)</th>
            <th style='padding:8px;border-bottom:1px solid #d1d5db;width:12%'>Function Call(s)</th>
          </tr>
        </thead>
        <tbody>
          {''.join(row_html)}
        </tbody>
      </table>
    </div>
    """
    display(HTML(table))

    # Footer sections
    _render_kv_table("Metadata", meta)
    _render_kv_table("Outcome", outcome)

    ann_rows = []
    for name, ann in [("an1", an1_row), ("an2", an2_row), ("an3", an3_row)]:
        if ann is None:
            ann_rows.append({"annotator": name, "present": False, "quality_score": None, "risk_flags": None, "failure_points_count": None})
        else:
            ann_rows.append(
                {
                    "annotator": name,
                    "present": True,
                    "quality_score": ann.get("quality_score"),
                    "risk_flags": ann.get("risk_flags"),
                    "failure_points_count": len(ann.get("failure_points", []) or []),
                }
            )

    ann_html_rows = []
    for r in ann_rows:
        ann_html_rows.append(
            "<tr>"
            f"<td style='padding:6px 8px;border-bottom:1px solid #e5e7eb'>{html.escape(str(r['annotator']))}</td>"
            f"<td style='padding:6px 8px;border-bottom:1px solid #e5e7eb'>{html.escape(str(r['present']))}</td>"
            f"<td style='padding:6px 8px;border-bottom:1px solid #e5e7eb'>{html.escape(str(r['quality_score']))}</td>"
            f"<td style='padding:6px 8px;border-bottom:1px solid #e5e7eb'>{html.escape(str(r['risk_flags']))}</td>"
            f"<td style='padding:6px 8px;border-bottom:1px solid #e5e7eb'>{html.escape(str(r['failure_points_count']))}</td>"
            "</tr>"
        )

    ann_table = f"""
    <div style='margin-top:10px'><b>Annotations</b></div>
    <div style='border:1px solid #d1d5db;border-radius:8px;overflow:auto;'>
      <table style='width:100%;border-collapse:collapse;font-size:13px;'>
        <thead style='background:#f9fafb;'>
          <tr>
            <th style='padding:6px 8px;border-bottom:1px solid #d1d5db'>Annotator</th>
            <th style='padding:6px 8px;border-bottom:1px solid #d1d5db'>Present</th>
            <th style='padding:6px 8px;border-bottom:1px solid #d1d5db'>Quality</th>
            <th style='padding:6px 8px;border-bottom:1px solid #d1d5db'>Risk Flags</th>
            <th style='padding:6px 8px;border-bottom:1px solid #d1d5db'>Failure Points</th>
          </tr>
        </thead>
        <tbody>
          {''.join(ann_html_rows)}
        </tbody>
      </table>
    </div>
    """
    display(HTML(ann_table))


print("Clean visualizer ready: render_conversation_clean('<conversation_id>')")

In [ ]:
# 20) Example usage (clean view)

render_conversation_clean("9e718fe5-44d1-6f6e-b794-6a764c7603ee")

In [83]:
# 21) Fixed Visualizer: Card-Per-Turn Layout (v3)
# Replaces render_conversation_clean. Call: show_conversation("<id>")

import html as _html
from IPython.display import display, HTML


STATE_ID_MAP = {
    "new": 0, "message_received": 1, "verification": 2, "intent_asked": 3,
    "settlement_explained": 4, "amount_pending": 5, "amount_sent": 6,
    "date_amount_asked": 7, "payment_confirmed": 8, "escalated": 9, "dormant": 10,
}

CLS_COLORS = {
    "unclear":            ("#fde68a", "#92400e"),
    "wants_settlement":   ("#d1fae5", "#065f46"),
    "wants_closure":      ("#cffafe", "#164e63"),
    "refuses":            ("#fee2e2", "#991b1b"),
    "disputes":           ("#ffe4e6", "#9f1239"),
    "hardship":           ("#fce7f3", "#9d174d"),
    "asks_time":          ("#e0e7ff", "#3730a3"),
}

CONF_COLORS = {
    "high":   ("#bbf7d0", "#14532d"),
    "medium": ("#fef3c7", "#78350f"),
    "low":    ("#fee2e2", "#7f1d1d"),
}


def _h(v):
    return _html.escape(str(v)) if v is not None else ""

def _annotate(s):
    if not isinstance(s, str):
        return str(s)
    sid = STATE_ID_MAP.get(s.strip())
    return f"{s}[{sid}]" if sid is not None else s

def _tiny_badge(text, bg="#f3f4f6", fg="#374151", bold=False):
    bw = "font-weight:600;" if bold else ""
    return (
        f"<span style='display:inline-block;padding:1px 7px;border-radius:4px;"
        f"background:{bg};color:{fg};font-size:11px;{bw}'>{_h(text)}</span>"
    )

def _pill(text, bg="#f3f4f6", fg="#374151"):
    return (
        f"<span style='display:inline-block;padding:2px 9px;border-radius:999px;"
        f"background:{bg};color:{fg};font-size:11px;margin:1px 2px 1px 0;line-height:1.4'>{_h(text)}</span>"
    )

def _kv_row(k, v, striped=False):
    bg = "background:#fafafa;" if striped else ""
    return (
        f"<tr style='{bg}'>"
        f"<td style='padding:4px 10px;border-bottom:1px solid #f0f0f0;color:#6b7280;font-size:12px;"
        f"white-space:nowrap;font-weight:600;vertical-align:top'>{_h(k)}</td>"
        f"<td style='padding:4px 10px;border-bottom:1px solid #f0f0f0;font-size:12px;word-break:break-word'>{_h(v)}</td>"
        f"</tr>"
    )


def _ensure_bundle():
    global conv_bundle
    if "conv_bundle" in globals() and isinstance(conv_bundle, dict) and conv_bundle:
        return conv_bundle
    log_map     = {x["conversation_id"]: x for x in logs}
    outcome_map = {x["conversation_id"]: x for x in outcomes}
    a1_map      = {x["conversation_id"]: x for x in a1}
    a2_map      = {x["conversation_id"]: x for x in a2}
    a3_map      = {x["conversation_id"]: x for x in a3}
    all_ids = sorted(set(log_map)|set(outcome_map)|set(a1_map)|set(a2_map)|set(a3_map))
    conv_bundle = {}
    for cid in all_ids:
        conv_bundle[cid] = dict(
            conversation_id=cid,
            prod_log=log_map.get(cid),
            outcome=outcome_map.get(cid),
            an1=a1_map.get(cid), an2=a2_map.get(cid), an3=a3_map.get(cid),
        )
    return conv_bundle


def _build_turns(prod_log):
    messages      = sorted(prod_log.get("messages", []) or [], key=lambda x: (x.get("turn",-1), x.get("timestamp","")))
    cls_map       = {c["turn"]: c for c in (prod_log.get("bot_classifications", []) or [])}
    trans_map     = {}
    for t in (prod_log.get("state_transitions", []) or []):
        trans_map.setdefault(t["turn"], []).append(t)
    fn_map        = {}
    for f in (prod_log.get("function_calls", []) or []):
        fn_map.setdefault(f["turn"], []).append(f)

    out = []
    for m in messages:
        t = m.get("turn")
        c = cls_map.get(t, {}) if m.get("role") == "borrower" else {}
        out.append(dict(
            turn=t,
            timestamp=m.get("timestamp",""),
            role=m.get("role",""),
            text=m.get("text",""),
            cls=c.get("classification"),
            conf=c.get("confidence"),
            transitions=trans_map.get(t, []),
            functions=fn_map.get(t, []),
        ))
    return out


def show_conversation(conversation_id: str):
    bundle = _ensure_bundle()
    row = bundle.get(conversation_id)
    if not row:
        print(f"Not found: {conversation_id}")
        return

    prod_log = row.get("prod_log")
    outcome  = row.get("outcome") or {}
    an1_r, an2_r, an3_r = row.get("an1"), row.get("an2"), row.get("an3")
    if not prod_log:
        print(f"No prod_log for: {conversation_id}")
        return

    meta  = prod_log.get("metadata", {}) or {}
    turns = _build_turns(prod_log)

    # ── Header ─────────────────────────────────────────────────────────────
    pr  = _pill(f"payment_received={outcome.get('payment_received')}", "#dcfce7", "#166534")
    cmp = _pill(f"complained={outcome.get('borrower_complained')}", "#fee2e2", "#991b1b")
    reg = _pill(f"regulatory_flag={outcome.get('regulatory_flag')}", "#fef3c7", "#92400e")
    att = _pill(f"channel_attribution={outcome.get('channel_attribution')}", "#e0f2fe", "#075985")

    header = f"""
    <div style='border:2px solid #d1d5db;border-radius:10px;padding:12px 16px;margin:0 0 12px 0;'>
      <div style='font-size:16px;font-weight:700;margin-bottom:6px;'>
        Conversation: <code style='font-size:14px'>{_h(conversation_id)}</code>
      </div>
      <div style='font-size:13px;line-height:1.8'>
        <b>Language:</b> {_h(meta.get('language'))} &nbsp;|&nbsp;
        <b>Zone:</b> {_h(meta.get('zone'))} &nbsp;|&nbsp;
        <b>DPD:</b> {_h(meta.get('dpd'))} &nbsp;|&nbsp;
        <b>POS / TOS:</b> ₹{_h(meta.get('pos'))} / ₹{_h(meta.get('tos'))} &nbsp;|&nbsp;
        <b>Total Turns:</b> {_h(meta.get('total_turns'))}
      </div>
      <div style='margin-top:6px'><b style='font-size:13px'>Outcome: </b>{pr}{cmp}{reg}{att}</div>
    </div>"""
    display(HTML(header))

    # ── State legend ───────────────────────────────────────────────────────
    legend = "  ".join(f"<b style='font-size:11px'>[{v}]</b> {_h(k)}" for k, v in STATE_ID_MAP.items())
    display(HTML(f"<div style='font-size:11px;color:#6b7280;margin:0 0 10px 0'><b>States: </b>{legend}</div>"))

    # ── Turn cards ─────────────────────────────────────────────────────────
    cards = []
    for r in turns:
        role = r["role"]

        # --- role badge
        if role == "bot":
            role_badge = "<span style='padding:1px 10px;border-radius:999px;background:#dbeafe;color:#1d4ed8;font-weight:700;font-size:12px'>bot</span>"
            row_bg     = "#f8faff"
        else:
            role_badge = "<span style='padding:1px 10px;border-radius:999px;background:#dcfce7;color:#166534;font-weight:700;font-size:12px'>borrower</span>"
            row_bg     = "#f8fff9"

        # --- turn + role + timestamp (left meta strip)
        ts = str(r["timestamp"]).replace("T", " ")
        left = (
            f"<div style='min-width:80px;width:80px;padding:8px 10px;border-right:2px solid #e5e7eb;"
            f"background:{row_bg};display:flex;flex-direction:column;align-items:center;gap:4px;"
            f"justify-content:flex-start'>"
            f"<span style='font-size:16px;font-weight:700;color:#374151'>{_h(r['turn'])}</span>"
            f"{role_badge}"
            f"<span style='font-size:10px;color:#9ca3af;margin-top:2px;text-align:center'>{_h(ts)}</span>"
            f"</div>"
        )

        # --- main text (center, dominant)
        text_block = f"<div style='font-size:13px;line-height:1.6;color:#111827;padding:10px 14px'>{_h(r['text'])}</div>"

        # --- right info panel (classification + transitions + functions)
        right_parts = []

        # classification + confidence
        if r.get("cls"):
            cls_bg, cls_fg = CLS_COLORS.get(r["cls"], ("#f3f4f6", "#374151"))
            conf_bg, conf_fg = CONF_COLORS.get(r.get("conf",""), ("#f3f4f6", "#374151"))
            right_parts.append(
                f"<div style='margin-bottom:5px'>"
                f"<span style='padding:2px 9px;border-radius:999px;background:{cls_bg};color:{cls_fg};"
                f"font-size:12px;font-weight:600'>{_h(r['cls'])}</span>"
                f"<span style='padding:2px 8px;border-radius:999px;background:{conf_bg};color:{conf_fg};"
                f"font-size:11px;margin-left:4px'>{_h(r.get('conf'))}</span>"
                f"</div>"
            )

        # transitions: each on its own line
        for t in r.get("transitions", []):
            src = _annotate(t.get("from_state",""))
            tgt = _annotate(t.get("to_state",""))
            right_parts.append(
                f"<div style='font-size:11px;font-family:monospace;color:#374151;"
                f"background:#f0fdf4;border:1px solid #bbf7d0;border-radius:4px;"
                f"padding:3px 8px;margin:2px 0;white-space:normal'>"
                f"{_h(src)} <b style='color:#059669'>→</b> {_h(tgt)}"
                f"</div>"
            )

        # function calls: each on its own line, params below
        for fn in r.get("functions", []):
            fname  = fn.get("function","")
            params = fn.get("params", {})
            param_str = "  ".join(f"{_h(k)}: {_h(v)}" for k, v in params.items())
            right_parts.append(
                f"<div style='font-size:11px;font-family:monospace;color:#1e1b4b;"
                f"background:#eef2ff;border:1px solid #c7d2fe;border-radius:4px;"
                f"padding:3px 8px;margin:2px 0'>"
                f"<b style='color:#4338ca'>{_h(fname)}</b><br>"
                f"<span style='color:#6366f1;font-size:10px'>{param_str}</span>"
                f"</div>"
            )

        right_panel = (
            f"<div style='min-width:260px;width:260px;padding:8px 10px;"
            f"border-left:2px solid #e5e7eb;background:{row_bg}'>"
            + ("".join(right_parts) if right_parts else "<span style='color:#d1d5db;font-size:11px'>—</span>")
            + "</div>"
        )

        card = (
            f"<div style='display:flex;border-bottom:1px solid #e5e7eb;background:{row_bg}'>"
            f"{left}"
            f"<div style='flex:1;min-width:0'>{text_block}</div>"
            f"{right_panel}"
            f"</div>"
        )
        cards.append(card)

    timeline = (
        "<div style='border:2px solid #d1d5db;border-radius:10px;overflow:hidden;margin-bottom:14px;'>"
        + "".join(cards) + "</div>"
    )
    display(HTML(timeline))

    # ── Metadata + Outcome ─────────────────────────────────────────────────
    meta_rows = "".join(_kv_row(k, v, i%2==1) for i,(k,v) in enumerate(meta.items()))
    out_items = {k:v for k,v in outcome.items() if k != "conversation_id"}
    out_rows  = "".join(_kv_row(k, v, i%2==1) for i,(k,v) in enumerate(out_items.items()))

    two_col = f"""
    <div style='display:flex;gap:14px;margin-bottom:14px;align-items:flex-start;'>
      <div style='flex:1;border:1px solid #d1d5db;border-radius:8px;overflow:hidden;'>
        <div style='padding:6px 10px;background:#f9fafb;font-weight:700;font-size:13px;border-bottom:1px solid #d1d5db'>Metadata</div>
        <table style='width:100%;border-collapse:collapse'>{meta_rows}</table>
      </div>
      <div style='flex:1;border:1px solid #d1d5db;border-radius:8px;overflow:hidden;'>
        <div style='padding:6px 10px;background:#f9fafb;font-weight:700;font-size:13px;border-bottom:1px solid #d1d5db'>Outcome</div>
        <table style='width:100%;border-collapse:collapse'>{out_rows}</table>
      </div>
    </div>"""
    display(HTML(two_col))

    # ── Annotations ────────────────────────────────────────────────────────
    ann_cells = []
    for label, ann in [("an1", an1_r), ("an2", an2_r), ("an3", an3_r)]:
        if ann is None:
            ann_cells.append(
                f"<div style='flex:1;border:1px solid #e5e7eb;border-radius:8px;padding:10px;"
                f"color:#9ca3af;font-size:12px;text-align:center'>{label}: not annotated</div>"
            )
        else:
            qs    = ann.get("quality_score")
            flags = ann.get("risk_flags") or []
            fps   = ann.get("failure_points") or []
            asmt  = ann.get("overall_assessment","")
            qs_bg = "#dcfce7" if qs and qs >= 0.7 else ("#fef3c7" if qs and qs >= 0.4 else "#fee2e2")
            qs_fg = "#166534" if qs and qs >= 0.7 else ("#92400e" if qs and qs >= 0.4 else "#991b1b")
            flags_html = " ".join(_tiny_badge(f, "#fee2e2", "#991b1b") for f in flags) if flags else "<span style='color:#9ca3af;font-size:11px'>none</span>"
            fp_rows = "".join(
                f"<tr><td style='padding:2px 6px;font-size:11px;color:#6b7280'>T{fp.get('turn','?')}</td>"
                f"<td style='padding:2px 6px;font-size:11px'>{_h(fp.get('category'))}</td>"
                f"<td style='padding:2px 6px;font-size:11px;color:#dc2626'>{fp.get('severity','')}</td>"
                f"<td style='padding:2px 6px;font-size:11px;color:#374151'>{_h(fp.get('note',''))}</td></tr>"
                for fp in fps
            )
            fp_table = (
                f"<table style='width:100%;border-collapse:collapse;margin-top:4px'>"
                f"<thead><tr style='background:#f9fafb'>"
                f"<th style='padding:2px 6px;font-size:11px;text-align:left'>Turn</th>"
                f"<th style='padding:2px 6px;font-size:11px;text-align:left'>Category</th>"
                f"<th style='padding:2px 6px;font-size:11px;text-align:left'>Sev</th>"
                f"<th style='padding:2px 6px;font-size:11px;text-align:left'>Note</th>"
                f"</tr></thead><tbody>{fp_rows}</tbody></table>"
            ) if fps else "<span style='font-size:11px;color:#9ca3af'>no failure points</span>"

            ann_cells.append(
                (
                    f"<div style='flex:1;border:1px solid #d1d5db;border-radius:8px;overflow:hidden;'>"
                    f"<div style='padding:6px 10px;background:#f9fafb;font-weight:700;font-size:13px;border-bottom:1px solid #d1d5db;display:flex;align-items:center;gap:8px'>"
                    f"{_h(label)}"
                    f"<span style='padding:2px 10px;border-radius:999px;background:{qs_bg};color:{qs_fg};font-size:13px;font-weight:700'>{qs}</span>"
                    f"</div>"
                    f"<div style='padding:8px 10px;font-size:12px'>"
                    f"<div style='margin-bottom:5px'><b>Risk flags:</b> {flags_html}</div>"
                    f"<div style='margin-bottom:5px;color:#374151;font-size:12px;font-style:italic'>{_h(asmt)}</div>"
                    f"<div><b>Failure points ({len(fps)}):</b> {fp_table}</div>"
                    f"</div></div>"
                )
            )
       

    # Fix: HTML() expects a single string, not multiple arguments.
    display(HTML(
        "<div style='font-weight:700;font-size:13px;margin-bottom:6px'>Annotations</div>" +
        f"<div style='display:flex;gap:14px;align-items:flex-start;'>{''.join(ann_cells)}</div>"
    ))


print("show_conversation('<conversation_id>') is ready")

show_conversation('<conversation_id>') is ready


In [85]:
# 22) Try it out

# show_conversation("051670b7-b772-dc86-433a-099a40b8ab18")
show_conversation("003d61b9-9929-0d01-78c0-72d65befe630")

language,hinglish
zone,west
dpd,97
pos,195000
tos,224250
settlement_offered,171500
total_turns,9
payment_received,True
days_to_payment,40
payment_amount,9245
expected_amount,10000


Turn,Category,Sev,Note
T6,repetition,0.8,"The bot repeats the same message from turn 5, indicating a failure to understand the borrower's response and adapt the conversation flow."
T8,context_loss,0.7,"Despite the borrower indicating they need time to think and will be contacted later, the bot asks about payment details again. It does not remember the prior responses."
T8,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
T5,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
T6,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
T7,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
T8,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
T8,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
T7,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
T5,repetition,0.9,"The bot gets stuck in a loop, repeating the same question about the payment date multiple times. It demonstrates a severe failure to understand the borrower's vague responses and adapt appropriately."
